In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    concat,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    when,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_real_estate_building_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #     "RealEstateBuildingId String, SourceSystem String, IngestionTimestamp timestamp, BatchId String, RealEstateBuildingKey string, RealEstateBuildingType string,"
        #     "RealEstateKey string, RealEstateCepalKey String, RealEstateStatusCode String, RealEstateStatus String, SourceTableName String, RealEstateComments String, RealEstateActive String,"
        #     "RealEstateDescription String, RealEstateSubCategoryCode String, RealEstateSubCategory String, RealEstateCategoryCode String, RealEstateCategory String,"
        #     "RealEstateAddress1 String, RealEstateAddress2 String, RealEstateBuildingNumber String, RealEstateEntranceAddress String, RealEstatePostalCode String,"
        #     "RealEstateGeolocation String, RealEstateLocationLambda double, RealEstateLocationPhi double, RealEstateEncumbrances String, RealEstateEncumbrancesStatus String, RealEstateEncumbrancesDetails String, "
        #     "RealEstateLandArea double, RealEstateContractArea double, RealEstateNetArea double, RealEstateGlaArea double,  RealEstateOccupancyTypeCode string, RealEstateOccupancyType string, "
        #     "RealEstateOccupancyRate double, RealEstateSurfaceArea double, RealEstatePlottedArea double,"
        #     "RealEstatePlottedDate date, RealEstateAcquisitionValue decimal(19,4), RealEstateBookValue decimal(19,4), RealEstateApprovedIASValue decimal(19,4), RealEstateFiscalValue decimal(19,4),"
        #     "RealEstateMarketValue decimal(19,4), RealEstateEvaluationDate date, RealEstateEvaluationTypeCode String, RealEstateEvaluationType String, RealEstateForcedSaleValue decimal(19,4),"
        #     "RealEstateSaleAmount decimal(19,4), RealEstateSaleVATAmount decimal(19,4), RealEstateSaleTotalAmount decimal(19,4), RealEstateSaleDate date, RealEstateDateEntered date,"
        #     "RealEstateFirstOfferPrice decimal(19,4), RealEstateMinimumPrice decimal(19,4), RealEstateMaximumPrice decimal(19,4), RealEstateAuctionAmount decimal(19,4), RealEstateConstructionPercentage String,"
        #     "RealEstateLegalIssuesFlag boolean, RealEstateValueDate date, RealEstateGroupMarketValueDate date, RealEstateValueTypeCode String, RealEstateValueType String, RealEstateGroupMarketValue decimal(19,2),"
        #     "RealEstateGroupMarketValueTypeCode String, RealEstateGroupMarketValueType String, RealEstateConstructionYear String, RealEstateLandRegistryCode String, RealEstateLandRegistryOffice String,"
        #     "RealEstateLandRegistry String, RealEstateUsageCodeTypeCode String, RealEstateUsageCodeType String, RealEstateSiteCode String, RealEstatePrimaryLandLocation String, RealEstateTotalTax decimal(19,2),"
        #     "RealEstateOperatingCost decimal(19,2), RealEstateArea String, RealEstateCity String, RealEstateIsEAPFlag boolean, RealEstateDistrict String, RealEstateDivision String,"
        #     "RealEstateMunicipality String, RealEstateCounty String, RealEstateMarketabilityCode String, RealEstateKaekCode String, RealEstateKaekOffice String, RealEstateHotelFlag boolean,"
        #     "RealEstateHotelStars String, RealEstateHotelRooms int, RealEstateSpecialConditions String, RealEstateAuctionDate timestamp, RealEstateAuctionCode String, RealEstatePlaceCode String,"
        #     "RealEstateConstructionStatusCode String, RealEstateConstructionStatus String, RealEstateBuildingName String, RealEstateBuildingActive String, RealEstateBuildingLandRegistryUnit String,"
        #     "RealEstateBuildingLandBookNumber String, RealEstateBuildingOldLandRegistryUnit String, RealEstatePropertyBuildingNumber String, RealEstateBuildingOldLandBookNumber String,"
        #     "RealEstateBuildingPreviousRegistryUnit String, RealEstateBuildingTopographicNumber String, RealEstateBuildingBuildingTypeCode String, RealEstateBuildingBuildingType String,"
        #     "RealEstateBuildingDescription String, RealEstateBuildingFloorRawValue double, RealEstateBuildingContractArea double, RealEstateBuildingNetArea double, RealEstateBuildingSurfaceArea2 double,"
        #     "RealEstateBuildingGrossRentableArea double, RealEstateBuildingOccupancyRate double, RealEstateBuildingRenovationYear int, RealEstateBuildingPermitNumber String, RealEstateLandOwnership string,"
        #     "RealEstateBuildingPermitDate timestamp, RealEstateBuildingFacadeArea double, RealEstateBuildingFreeHeight double, RealEstateBuildingTechnicalNotes String, RealEstateBuildingOtherAuxiliaryArea double,"
        #     "RealEstateBuildingParkingSpace int, RealEstateBuildingStorageArea double, RealEstateBuildingLoftArea double, RealEstateBuildingDetachmentArea String, RealEstateBuildingPowerNumber String,"
        #     "RealEstateBuildingDateEntered timestamp, RealEstateBuildingEnteredBy int, RealEstateBuildingAreaTitle double, RealEstateBuildingAreaLicense double, RealEstateBuildingAreaSettled double,"
        #     "RealEstateBuildingUsageCode String, RealEstateBuildingParkingArea double, RealEstateBuildingWarehouseNumber int, RealEstateIsEligibleFlag boolean, RealEstateCountry string,"
        #     "CreatedDate string, CreatedBy string, LastUpdatedDate string, LastUpdatedBy string,  LocationTrackingCode string, LocationTracking string, BusinessDate date, RealEstateBuildingFloorRawValueDescription STRING"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.RealEstateBuilding"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== eaiSIG_RE ==================

        eaiSIG_RE_extra_filters = None

        if eaiSIG_RE_extra_filters is not None:
            eaiSIG_RE_extra_filters.append(date_filter)
            eaiSIG_RE_final_filters = " AND ".join(eaiSIG_RE_extra_filters)
        else:
            eaiSIG_RE_final_filters = date_filter

        eaiSIG_RE_selected_fields = [
            "eaiSIG_RE_TOPONYM",
            "eaiSIG_RE_DISTRICT",
            "eaiSIG_RE_REAL_ESTATE_CODE",
            "eaiSIG_RE_COMMENTS",
            "eaiSIG_RE_ADDRESS",
            "eaiSIG_RE_POSTAL_CODE",
            "eaiSIG_RE_LAND_AREA",
            "eaiSIG_RE_AREA",
            "eaiSIG_RE_LAND_PERCENTAGE",
            "eaiSIG_RE_LAND_NUMBER",
            "eaiSIG_RE_LAND_REGISTRY",
            "eaiSIG_RE_PREFECTURE",
            "eaiSIG_RE_CITY",
            "eaiSIG_RE_MUNICIPALITY",
            "businessdate",
        ]

        eaiSIG_RE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_RE",
            eaiSIG_RE_final_filters,
            eaiSIG_RE_selected_fields,
        )

        # ================== eaiSIG_REPE ==================

        eaiSIG_REPE_extra_filters = None

        if eaiSIG_REPE_extra_filters is not None:
            eaiSIG_REPE_extra_filters.append(date_filter)
            eaiSIG_REPE_final_filters = " AND ".join(eaiSIG_REPE_extra_filters)
        else:
            eaiSIG_REPE_final_filters = date_filter

        eaiSIG_REPE_selected_fields = [
            "eaiSIG_REPE_REST_BOOK_VALUE",
            "eaiSIG_REPE_REST_OBJECTIVE_DATE",
            "eaiSIG_REPE_REST_MARKET_VALUE",
            "eaiSIG_REPE_REST_EVALUATION_DATE",
            "eaiSIG_REPE_REST_APPR_SOURCE",
            "eaiSIG_REPE_REST_LEG_STATUS",
            "eaiSIG_REPE_REAL_ESTATE_CODE",
            "businessdate",
        ]

        eaiSIG_REPE = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "arotron_eaiSIG_REPE",
            eaiSIG_REPE_final_filters,
            eaiSIG_REPE_selected_fields,
        )

        eaiSIG_RE_eaiSIG_REPE_join_condition = (
            eaiSIG_RE["eaiSIG_RE_REAL_ESTATE_CODE"]
            == eaiSIG_REPE["eaiSIG_REPE_REAL_ESTATE_CODE"]
        )

        # Joins
        target_table_arotron = eaiSIG_RE.join(
            eaiSIG_REPE, eaiSIG_RE_eaiSIG_REPE_join_condition, "LEFT"
        )

        # HashKeys
        target_table_arotron_RealEstateBuildingId_primary_key_fields = [
            eaiSIG_RE.eaiSIG_RE_REAL_ESTATE_CODE,
            lit("arotron"),
        ]

        target_table_arotron = (
            target_table_arotron.withColumn(
                "RealEstateBuildingId",
                sha2(
                    concat_ws(
                        ";",
                        *target_table_arotron_RealEstateBuildingId_primary_key_fields,
                    ),
                    256,
                ),
            )
            .withColumn(
                "RealEstateAddress2",
                when(
                    eaiSIG_RE.eaiSIG_RE_DISTRICT != eaiSIG_RE.eaiSIG_RE_TOPONYM,
                    concat(
                        eaiSIG_RE.eaiSIG_RE_DISTRICT,
                        lit(" "),
                        eaiSIG_RE.eaiSIG_RE_TOPONYM,
                    ),
                ).otherwise(eaiSIG_RE.eaiSIG_RE_DISTRICT),
            )
            .withColumn(
                "RealEstateLandArea",
                when(eaiSIG_RE.eaiSIG_RE_LAND_AREA == "", lit(None)).otherwise(
                    eaiSIG_RE.eaiSIG_RE_LAND_AREA.cast("double")
                ),
            )
            .withColumn(
                "RealEstateNetArea",
                when(eaiSIG_RE.eaiSIG_RE_AREA == "", lit(None)).otherwise(
                    eaiSIG_RE.eaiSIG_RE_AREA.cast("double")
                ),
            )
            .withColumn(
                "RealEstateOccupancyRate",
                when(eaiSIG_RE.eaiSIG_RE_LAND_PERCENTAGE == "", lit(None)).otherwise(
                    eaiSIG_RE.eaiSIG_RE_LAND_PERCENTAGE.cast("double")
                ),
            )
            .withColumn(
                "RealEstatePlottedArea",
                when(eaiSIG_RE.eaiSIG_RE_LAND_AREA == "", lit(None)).otherwise(
                    eaiSIG_RE.eaiSIG_RE_LAND_AREA.cast("double")
                ),
            )
            .withColumn(
                "RealEstateBookValue",
                when(eaiSIG_REPE.eaiSIG_REPE_REST_BOOK_VALUE == "", -9999).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_BOOK_VALUE.cast("decimal(19,4)")
                ),
            )
            .withColumn(
                "RealEstateFiscalValue",
                when(
                    eaiSIG_REPE.eaiSIG_REPE_REST_OBJECTIVE_DATE == "", -9999
                ).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_OBJECTIVE_DATE.cast("decimal(19,4)")
                ),
            )
            .withColumn(
                "RealEstateMarketValue",
                when(eaiSIG_REPE.eaiSIG_REPE_REST_MARKET_VALUE == "", -9999).otherwise(
                    eaiSIG_REPE.eaiSIG_REPE_REST_MARKET_VALUE.cast("decimal(19,4)")
                ),
            )
            .withColumn(
                "RealEstateEvaluationDate",
                when(
                    eaiSIG_REPE.eaiSIG_REPE_REST_EVALUATION_DATE == "", lit(None)
                ).otherwise(eaiSIG_REPE.eaiSIG_REPE_REST_EVALUATION_DATE.cast("date")),
            )
        )

        # ================== Selection of Final Columns ==================

        result_df_arotron = target_table_arotron.select(
            target_table_arotron.RealEstateBuildingId.cast("String").alias(
                "RealEstateBuildingId"
            ),
            lit("arotron").cast("String").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit(None).cast("String").alias("BatchId"),
            eaiSIG_RE.eaiSIG_RE_REAL_ESTATE_CODE.cast("string").alias(
                "RealEstateBuildingKey"
            ),
            lit("Real Estate").alias("RealEstateBuildingType"),
            eaiSIG_RE.eaiSIG_RE_REAL_ESTATE_CODE.cast("String").alias("RealEstateKey"),
            lit(None).cast("String").alias("RealEstateCepalKey"),
            lit(None).cast("String").alias("RealEstateStatusCode"),
            lit(None).cast("String").alias("RealEstateStatus"),
            lit(None).cast("String").alias("SourceTableName"),
            eaiSIG_RE.eaiSIG_RE_COMMENTS.cast("String").alias("RealEstateComments"),
            lit(None).cast("String").alias("RealEstateActive"),
            lit(None).cast("String").alias("RealEstateDescription"),
            lit(None).cast("String").alias("RealEstateSubCategoryCode"),
            lit(None).cast("String").alias("RealEstateSubCategory"),
            lit(None).cast("String").alias("RealEstateCategoryCode"),
            lit(None).cast("String").alias("RealEstateCategory"),
            lit(None).cast("string").alias("RealEstateInsuranceCategory"),
            eaiSIG_RE.eaiSIG_RE_ADDRESS.cast("String").alias("RealEstateAddress1"),
            target_table_arotron.RealEstateAddress2.alias("RealEstateAddress2"),
            lit(None).cast("String").alias("RealEstateBuildingNumber"),
            lit(None).cast("String").alias("RealEstateEntranceAddress"),
            eaiSIG_RE.eaiSIG_RE_POSTAL_CODE.cast("String").alias(
                "RealEstatePostalCode"
            ),
            lit(None).cast("String").alias("RealEstateGeolocation"),
            lit(None).cast("double").alias("RealEstateLocationLambda"),
            lit(None).cast("double").alias("RealEstateLocationPhi"),
            lit(None).cast("String").alias("RealEstateEncumbrances"),
            lit(None).cast("String").alias("RealEstateEncumbrancesStatus"),
            lit(None).cast("String").alias("RealEstateEncumbrancesDetails"),
            target_table_arotron.RealEstateLandArea.alias(
                "RealEstateLandArea"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to double works. The blank values though get converted to 0.
            lit(None).cast("double").alias("RealEstateContractArea"),
            target_table_arotron.RealEstateNetArea.alias(
                "RealEstateNetArea"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to double works. The blank values though get converted to 0.
            lit(None).cast("double").alias("RealEstateGlaArea"),
            lit(None).cast("String").alias("RealEstateOccupancyTypeCode"),
            lit(None).cast("String").alias("RealEstateOccupancyType"),
            target_table_arotron.RealEstateOccupancyRate.alias(
                "RealEstateOccupancyRate"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to double works. The blank values though get converted to 0.
            lit(None).cast("double").alias("RealEstateSurfaceArea"),
            target_table_arotron.RealEstatePlottedArea.alias(
                "RealEstatePlottedArea"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to double works. The blank values though get converted to 0.
            lit(None).cast("date").alias("RealEstatePlottedDate"),
            lit(None).cast("decimal(19,4)").alias("RealEstateAcquisitionValue"),
            target_table_arotron.RealEstateBookValue.alias(
                "RealEstateBookValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            lit(None).cast("decimal(19,4)").alias("RealEstateApprovedIASValue"),
            target_table_arotron.RealEstateFiscalValue.alias(
                "RealEstateFiscalValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            target_table_arotron.RealEstateMarketValue.alias(
                "RealEstateMarketValue"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to decimal(19,4) works for non-blank values. For the blank values this doesn't work.
            target_table_arotron.RealEstateEvaluationDate.alias(
                "RealEstateEvaluationDate"
            ),  # problem, the field is actually of the "String" data type for the arotron source system, but the convertion to date works. The blank values though get converted to 1900-01-01.
            eaiSIG_REPE.eaiSIG_REPE_REST_APPR_SOURCE.cast("String").alias(
                "RealEstateEvaluationTypeCode"
            ),
            lit(None).cast("String").alias("RealEstateEvaluationType"),
            lit(None).cast("decimal(19,4)").alias("RealEstateForcedSaleValue"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleAmount"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleVATAmount"),
            lit(None).cast("decimal(19,4)").alias("RealEstateSaleTotalAmount"),
            lit(None).cast("date").alias("RealEstateSaleDate"),
            lit(None).cast("date").alias("RealEstateDateEntered"),
            lit(None).cast("decimal(19,4)").alias("RealEstateFirstOfferPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateMinimumPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateMaximumPrice"),
            lit(None).cast("decimal(19,4)").alias("RealEstateAuctionAmount"),
            lit(None).cast("String").alias("RealEstateConstructionPercentage"),
            eaiSIG_REPE.eaiSIG_REPE_REST_LEG_STATUS.cast("boolean").alias(
                "RealEstateLegalIssuesFlag"
            ),
            lit(None).cast("date").alias("RealEstateValueDate"),
            lit(None).cast("date").alias("RealEstateGroupMarketValueDate"),
            lit(None).cast("String").alias("RealEstateValueTypeCode"),
            lit(None).cast("String").alias("RealEstateValueType"),
            lit(None).cast("decimal(19,2)").alias("RealEstateGroupMarketValue"),
            lit(None).cast("String").alias("RealEstateGroupMarketValueTypeCode"),
            lit(None).cast("String").alias("RealEstateGroupMarketValueType"),
            lit(None).cast("String").alias("RealEstateConstructionYear"),
            eaiSIG_RE.eaiSIG_RE_LAND_NUMBER.cast("String").alias(
                "RealEstateLandRegistryCode"
            ),
            lit(None).cast("String").alias("RealEstateLandRegistryOffice"),
            eaiSIG_RE.eaiSIG_RE_LAND_REGISTRY.cast("String").alias(
                "RealEstateLandRegistry"
            ),
            lit(None).cast("String").alias("RealEstateUsageCodeTypeCode"),
            lit(None).cast("String").alias("RealEstateUsageCodeType"),
            lit(None).cast("String").alias("RealEstateSiteCode"),
            eaiSIG_RE.eaiSIG_RE_PREFECTURE.cast("String").alias(
                "RealEstatePrimaryLandLocation"
            ),
            lit(None).cast("decimal(19,2)").alias("RealEstateTotalTax"),
            lit(None).cast("decimal(19,2)").alias("RealEstateOperatingCost"),
            eaiSIG_RE.eaiSIG_RE_PREFECTURE.cast("String").alias("RealEstateArea"),
            eaiSIG_RE.eaiSIG_RE_CITY.cast("String").alias("RealEstateCity"),
            lit(None).cast("boolean").alias("RealEstateIsEAPFlag"),
            lit(None).cast("String").alias("RealEstateDistrict"),
            eaiSIG_RE.eaiSIG_RE_DISTRICT.cast("String").alias("RealEstateDivision"),
            eaiSIG_RE.eaiSIG_RE_MUNICIPALITY.cast("String").alias(
                "RealEstateMunicipality"
            ),
            lit(None).cast("String").alias("RealEstateCounty"),
            lit(None).cast("String").alias("RealEstateMarketabilityCode"),
            lit(None).cast("String").alias("RealEstateKaekCode"),
            lit(None).cast("String").alias("RealEstateKaekOffice"),
            lit(None).cast("boolean").alias("RealEstateHotelFlag"),
            lit(None).cast("String").alias("RealEstateHotelStars"),
            lit(None).cast("int").alias("RealEstateHotelRooms"),
            lit(None).cast("String").alias("RealEstateSpecialConditions"),
            lit(None).cast("timestamp").alias("RealEstateAuctionDate"),
            lit(None).cast("String").alias("RealEstateAuctionCode"),
            lit(None).cast("String").alias("RealEstatePlaceCode"),
            lit(None).cast("String").alias("RealEstateConstructionStatusCode"),
            lit(None).cast("String").alias("RealEstateConstructionStatus"),
            lit(None).cast("string").alias("RealEstateLandOwnership"),
            lit(None).cast("String").alias("RealEstateBuildingName"),
            lit(None).cast("String").alias("RealEstateBuildingActive"),
            lit(None).cast("String").alias("RealEstateBuildingLandRegistryUnit"),
            lit(None).cast("String").alias("RealEstateBuildingLandBookNumber"),
            lit(None).cast("String").alias("RealEstateBuildingOldLandRegistryUnit"),
            lit(None).cast("String").alias("RealEstatePropertyBuildingNumber"),
            lit(None).cast("String").alias("RealEstateBuildingOldLandBookNumber"),
            lit(None).cast("String").alias("RealEstateBuildingPreviousRegistryUnit"),
            lit(None).cast("String").alias("RealEstateBuildingTopographicNumber"),
            lit(None).cast("String").alias("RealEstateBuildingBuildingTypeCode"),
            lit(None).cast("String").alias("RealEstateBuildingBuildingType"),
            lit(None).cast("String").alias("RealEstateBuildingDescription"),
            lit(None).cast("double").alias("RealEstateBuildingFloorRawValue"),
            lit(None).cast("double").alias("RealEstateBuildingContractArea"),
            lit(None).cast("double").alias("RealEstateBuildingNetArea"),
            lit(None).cast("double").alias("RealEstateBuildingSurfaceArea2"),
            lit(None).cast("double").alias("RealEstateBuildingGrossRentableArea"),
            lit(None).cast("double").alias("RealEstateBuildingOccupancyRate"),
            lit(None).cast("int").alias("RealEstateBuildingRenovationYear"),
            lit(None).cast("String").alias("RealEstateBuildingPermitNumber"),
            lit(None).cast("timestamp").alias("RealEstateBuildingPermitDate"),
            lit(None).cast("double").alias("RealEstateBuildingFacadeArea"),
            lit(None).cast("double").alias("RealEstateBuildingFreeHeight"),
            lit(None).cast("String").alias("RealEstateBuildingTechnicalNotes"),
            lit(None).cast("double").alias("RealEstateBuildingOtherAuxiliaryArea"),
            lit(None).cast("int").alias("RealEstateBuildingParkingSpace"),
            lit(None).cast("double").alias("RealEstateBuildingStorageArea"),
            lit(None).cast("double").alias("RealEstateBuildingLoftArea"),
            lit(None).cast("String").alias("RealEstateBuildingDetachmentArea"),
            lit(None).cast("String").alias("RealEstateBuildingPowerNumber"),
            lit(None).cast("timestamp").alias("RealEstateBuildingDateEntered"),
            lit(None).cast("int").alias("RealEstateBuildingEnteredBy"),
            lit(None).cast("double").alias("RealEstateBuildingAreaTitle"),
            lit(None).cast("double").alias("RealEstateBuildingAreaLicense"),
            lit(None).cast("double").alias("RealEstateBuildingAreaSettled"),
            lit(None).cast("String").alias("RealEstateBuildingUsageCode"),
            lit(None).cast("double").alias("RealEstateBuildingParkingArea"),
            lit(None).cast("int").alias("RealEstateBuildingWarehouseNumber"),
            lit(None).cast("boolean").alias("RealEstateIsEligibleFlag"),
            lit(None).cast("string").alias("RealEstateCountry"),
            lit(None).cast("string").alias("CreatedDate"),
            lit(None).cast("string").alias("CreatedBy"),
            lit(None).cast("string").alias("LastUpdatedDate"),
            lit(None).cast("string").alias("LastUpdatedBy"),
            lit(None).cast("string").alias("LocationTrackingCode"),
            lit(None).cast("string").alias("LocationTracking"),
            eaiSIG_RE.businessdate.cast("date").alias("BusinessDate"),
            lit(None).cast("string").alias("RealEstateBuildingFloorRawValueDescription"),

        )

        result_df_arotron.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_real_estate_building_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)